In [12]:
%load_ext autoreload
%autoreload 2

#set the path to the root of the project dynamically
import sys
import os

os.chdir(os.path.join(os.getcwd(), os.pardir, os.pardir))  # Move up two levels to the project root

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
print("Current working directory:", os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir)))

Current working directory: /home/alexis/Documents/projets/GEM/gem_working_group


In [ ]:
from src.gem.engine.ecosystem_engine import EcosystemEngine
from src.gem.engine.environment_state import EnvironmentState
from src.gem.engine.ecosystem_grid_state import EcosystemGridState
from src.gem.engine.species_registry import SpeciesRegistry

from src.gem.engine.processes import apply_metabolism_mass_specific
import numpy as np
import rasterio

#load fromt he root of the project
processed_dir = "./data/processed"
temp_path = os.path.join(processed_dir, 'Tavg_NorthAmerica_EqualArea_100km_month_01.tif') # January example

# 2. Load the actual raster data
    
with rasterio.open(temp_path) as src:
    temp_data = src.read(1)
    
    

world_shape = (75, 120)  # (12000 km x 7500 km with 100 km cell size)
env = EnvironmentState(world_shape)
env.add_layer("temperature", temp_data)

print(f"Environment initialized with layers: {list(env.layers.keys())}")
print(f"Grid shape: {env.shape}")

# Create 3 species: 1 plant, 2 animals
registry = SpeciesRegistry(num_species=25)
registry.add_species_to_group(
    group_name="deciduous_trees", 
    species_indices=list(range(0, 5)), 
    params={
        "base_growth_rate": 0.15, 
        "mortality_rate": 0.05,
        "max_dispersal_rate": 0.01
    }
)

registry.add_species_to_group(
    group_name="coniferous_trees", 
    species_indices=list(range(10, 15)), 
    params={
        "base_growth_rate": 0.08, 
        "mortality_rate": 0.02,
        "max_dispersal_rate": 0.02
    }
)

registry.add_species_to_group(
    group_name="herbs", 
    species_indices=list(range(5, 10)), 
    params={
        "base_growth_rate": 0.2, 
        "mortality_rate": 0.1,
        "max_dispersal_rate": 0.05
    }
)
registry.add_species_to_group(
    group_name="herbivores", 
    species_indices=list(range(15, 18)), 
    params={
        "base_growth_rate": 0.1, 
        "mortality_rate": 0.05,
        "max_dispersal_rate": 0.1,
        "mass_g": 136 * 1000,  # Average mass of a deer in grams
        "is_endothermic": True,
        "is_mammal": True,
        "c_int": 19.53,
        "b": 0.73
    }
)
registry.add_species_to_group(
    group_name="carnivores", 
    species_indices=list(range(18, 20)), 
    params={
        "mass_g": 150 * 1000,  # Average mass of a wolf in grams
        "is_endothermic": True,
        "is_mammal": True,
        "c_int": 19.53,
        "b": 0.73
    }
)
registry.add_species_to_group(
    group_name="lizards", 
    species_indices=list(range(20, 25)), 
    params={
        "mass_g": 0.5 * 1000,
        "is_endothermic": False,
        "is_mammal": False,
        "c_int": 17.4,
        "b": 0.84
    }
)

#bigger groups for more interesting dynamics
registry.add_species_to_group(group_name="vertebrates", species_indices=list(range(15, 25))) # All vertebrates for process targeting
registry.add_species_to_group(group_name="plants", species_indices=list(range(0, 15))) # All plants for process targeting

#add feeding links (plants -> herbivores -> carnivores)
registry.add_feeding_link("plants", "herbivores")
registry.add_feeding_link("herbivores", "carnivores")


#Grid state with 20 species (15 plants, 5 animals)
grid = EcosystemGridState(world_shape, registry)

#ADDING LAYERS
grid.add_layer("metabolism_mass_specific")

#ADDING DELTA LAYERS
grid.add_delta_layer("metabolism_delta", source_layer="biomass")  # Register a delta for net growth rates

# SEED INITIAL LAYERS
grid.layers["biomass"][:, :, registry.get_group_indices("plants")] = 50000.0  # Plants
grid.layers["biomass"][:, :, registry.get_group_indices("animals")] = 5000.0  # Animals

# --- 2. Build the Engine ---
model = EcosystemEngine(grid, env)

# The order here defines the pipeline sequence
model.add_process(apply_metabolism_mass_specific)

# --- 3. Run the Simulation ---
num_steps = 2
vertebrate_biomass_history = []
plant_biomass_history = []
metabolism_history = []
for tick in range(num_steps):
    model.step()
    
    # Track Totals using the new matrix views
    # np.sum() cleanly aggregates the whole grid
    total_plant = np.sum(grid.get_layer_view("biomass", "plants"))
    total_vertebrates = np.sum(grid.get_layer_view("biomass", "vertebrates"))
    
    plant_biomass_history.append(total_plant)
    vertebrate_biomass_history.append(total_vertebrates)
    metabolism_mass_specific = grid.get_layer_view("metabolism_mass_specific", "vertebrates")
    metabolism_history.append(grid.get_layer_view("metabolism_mass_specific", "vertebrates"))
    


Environment initialized with layers: ['temperature']
Grid shape: (75, 120)
metabolis_mass_specific: [[[3.9091319e-05 3.9091319e-05 3.9091319e-05 ... 4.1807358e-05
   4.1807358e-05 4.1807358e-05]
  [3.9246876e-05 3.9246876e-05 3.9246876e-05 ... 4.1973723e-05
   4.1973723e-05 4.1973723e-05]
  [          nan           nan           nan ...           nan
             nan           nan]
  ...
  [4.1980154e-05 4.1980154e-05 4.1980154e-05 ... 4.4896908e-05
   4.4896908e-05 4.4896908e-05]
  [5.3894211e-05 5.3894211e-05 5.3894211e-05 ... 5.7638747e-05
   5.7638747e-05 5.7638747e-05]
  [6.7613648e-05 6.7613648e-05 6.7613648e-05 ... 7.2311406e-05
   7.2311406e-05 7.2311406e-05]]

 [[          nan           nan           nan ...           nan
             nan           nan]
  [          nan           nan           nan ...           nan
             nan           nan]
  [4.7245521e-05 4.7245521e-05 4.7245521e-05 ... 5.0528110e-05
   5.0528110e-05 5.0528110e-05]
  ...
  [4.7297122e-05 4.7297122e-05 

In [61]:
metabolism_history[1]

array([[[3.9091319e-05, 3.9091319e-05, 3.9091319e-05, ...,
         4.1807358e-05, 4.1807358e-05, 4.1807358e-05],
        [3.9246876e-05, 3.9246876e-05, 3.9246876e-05, ...,
         4.1973723e-05, 4.1973723e-05, 4.1973723e-05],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        ...,
        [4.1980154e-05, 4.1980154e-05, 4.1980154e-05, ...,
         4.4896908e-05, 4.4896908e-05, 4.4896908e-05],
        [5.3894211e-05, 5.3894211e-05, 5.3894211e-05, ...,
         5.7638747e-05, 5.7638747e-05, 5.7638747e-05],
        [6.7613648e-05, 6.7613648e-05, 6.7613648e-05, ...,
         7.2311406e-05, 7.2311406e-05, 7.2311406e-05]],

       [[          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [4.7245521e-05, 4.7245521e-05, 4.7245521e-05, ...,
         5.052